# AI Agents with LangChain & Gemma 3 4B
### A Hands-On Session for Beginners

---

## What You Will Learn

By the end of this session you will be able to:
- Understand what an **AI Agent** is and how it differs from a regular chatbot
- Understand the **ReAct** reasoning pattern (Think → Act → Observe)
- Load a real open-source LLM (**Gemma 3 4B**) from HuggingFace
- Give the agent a **Tool** (DuckDuckGo web search)
- Watch the agent **reason step by step** to answer a question

---

## Table of Contents
1. What is an AI Agent?
2. What is ReAct?
3. Environment Setup
4. Load Gemma 3 4B
5. Give the Agent a Tool — DuckDuckGo Search
6. Build the ReAct Agent
7. Run the Agent
8. Exercises

> **Runtime:** Make sure you have selected **Runtime > Change runtime type > T4 GPU** in Google Colab before running any cells.

---
## 1. What is an AI Agent?

A regular **chatbot** only responds to your message using knowledge baked in during training. It cannot go out and look things up, run code, or take actions in the real world.

An **AI Agent** is different. It is an LLM that has been given:
- A **goal** (your question or task)
- A set of **Tools** it can call (e.g. web search, calculator, database)
- A **loop** that lets it keep acting until it has a final answer

```
User asks a question
       |
       v
  [ LLM Thinks ]
       |
       v
  Does it need more info?
   Yes --> Calls a Tool --> Gets result --> Thinks again
   No  --> Returns Final Answer
```

**Analogy:** Think of the agent as a detective. It has a case (your question), and it goes out and collects evidence (tool calls) until it can solve the case (final answer).

---
## 2. What is ReAct?

**ReAct** = **Re**asoning + **Act**ing — a popular pattern for building agents.

The LLM is instructed to always output its thoughts in this structured format:

```
Question: Who won the 2024 Cricket World Cup?

Thought: I don't know the answer from memory, I should search the web.
Action: DuckDuckGoSearchRun
Action Input: "2024 Cricket World Cup winner"
Observation: India won the 2024 ICC T20 World Cup, defeating South Africa.

Thought: I now have the answer.
Final Answer: India won the 2024 ICC T20 Cricket World Cup.
```

LangChain automatically:
1. Parses the `Action` and `Action Input` from the model output
2. Runs the tool
3. Feeds the `Observation` back to the model
4. Repeats until `Final Answer` appears

This loop is managed by the **AgentExecutor**.

---
## 3. Environment Setup

### Step 3.1 — Check Your GPU

Gemma 3 4B is a large model. We need a GPU to run it fast enough to be useful. The cell below confirms your Colab GPU is active.

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

Fri May  1 03:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 3.2 — Install Required Libraries

We install four groups of packages:

| Package | Purpose |
|---|---|
| `langchain`, `langchain-community`, `langchain-huggingface` | The agent framework utilities |
| `transformers`, `accelerate` | Load and run HuggingFace models |
| `bitsandbytes` | 4-bit quantization to fit Gemma into Colab's 15 GB GPU RAM |
| `duckduckgo-search` | DuckDuckGo web search (no API key needed) |

> This cell takes about 2-3 minutes. Run it and wait for it to finish before continuing.

In [2]:
# Core framework — no version pins needed with the manual-loop approach
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q transformers accelerate bitsandbytes
!pip install -q ddgs huggingface_hub      # ddgs is the renamed duckduckgo-search package
print("All packages installed!")

All packages installed!


### Step 3.3 — Import Libraries

We import everything we need upfront so we don't interrupt the flow later.

In [3]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_community.tools import DuckDuckGoSearchRun

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


---
## Step 3.4 — Log in to HuggingFace

Gemma 3 is a **gated model** — you need to:
1. Have a [HuggingFace account](https://huggingface.co)
2. Visit [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) and accept the license
3. Create a **read token** at https://huggingface.co/settings/tokens
4. Paste it in the cell below when prompted

Your token is used only to download the model weights — it is not stored anywhere.

In [4]:
from huggingface_hub import login

# This will show a prompt where you paste your HuggingFace token
login("")

---
## 4. Load Gemma 3 4B

### What is 4-bit Quantization?

Gemma 3 4B in full precision (float32) would need ~16 GB of GPU RAM — too much for the free Colab T4.

**Quantization** compresses the model weights to 4-bit integers, reducing memory to ~3-4 GB with very little quality loss. We use the `bitsandbytes` library for this.

```
Full precision (float32):  ~16 GB  ❌ Too big for T4
Half precision (float16):   ~8 GB  ⚠️  Borderline
4-bit quantized:            ~3 GB  ✅  Fits comfortably
```

> This cell downloads ~2-3 GB from HuggingFace. It takes **3-5 minutes**. Wait for the progress bar to complete.

In [5]:
MODEL_ID = "google/gemma-3-4b-it"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NF4 is best for LLMs
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computations
    bnb_4bit_use_double_quant=True        # Extra compression with minimal quality loss
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this takes a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"   # Automatically place layers on GPU/CPU
)

print(f"\nModel loaded! Memory used: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]


Model loaded! Memory used: 3.17 GB


### Wrap the Model in a LangChain-Compatible Pipeline

LangChain cannot talk to a raw HuggingFace model directly. We use:
1. `pipeline(...)` — wraps the model into a simple text-generation interface
2. `HuggingFacePipeline(...)` — wraps that into something LangChain can call as an LLM

**Key parameters:**
- `max_new_tokens=1024` — maximum number of tokens the model can generate per call
- `temperature=0.1` — low temperature = more focused, less random output (good for agents)
- `return_full_text=False` — return only the generated text, not the input prompt

In [6]:
# Create a text-generation pipeline
hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
    repetition_penalty=1.1
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("LLM is ready!")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM is ready!


### Quick Sanity Test

Before building the agent, let us check that the LLM works correctly on its own. We ask it a simple question directly.

In [7]:
response = llm.invoke("What is the capital of France? Answer in one sentence.")
print("Model response:", response)

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model response: 

The capital of France is Paris.



---
## 5. Give the Agent a Tool — DuckDuckGo Search

### What is a Tool?

A **Tool** is any function the agent can decide to call. LangChain tools have:
- A **name** — used by the agent to refer to the tool in its reasoning
- A **description** — tells the agent *when* to use this tool
- A **function** — the actual code that runs

We use **DuckDuckGo Search** because:
- It is completely free — no API key needed
- It searches the live web — so results are always up to date
- LangChain has a built-in wrapper for it

In [8]:
# Create the DuckDuckGo search tool
search_tool = DuckDuckGoSearchRun()

# Inspect what the tool looks like to the agent
print("Tool name       :", search_tool.name)
print("Tool description:", search_tool.description)

Tool name       : duckduckgo_search
Tool description: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.


### Test the Search Tool Directly

Before plugging the tool into an agent, let's see what raw search results look like. The agent will receive exactly this kind of text as an `Observation`.

In [9]:
# Call the tool manually (no agent involved yet)
result = search_tool.run("latest AI models released in 2025")
print("Search result:")
print("-" * 60)
print(result[:800], "...")  # Show first 800 chars

Search result:
------------------------------------------------------------
2 weeks ago - 22 May – Anthropic releases Claude 4, with two models: Claude Opus 4 and Claude Sonnet 4. According to Anthropic, Claude 4 can function on its own for hours. 5 August – xAI launches their image generator Grok Imagine, which has a 'spicy mode' that allows users to create NSFW content. 1 week ago - November 2025 was extremely packed with new AI model releases, as GPT-5.1, Grok 4.1, Gemini 3 Pro, Claude Opus 4.5 all released within just 6 days of each other — if you’re looking for the best AI model for coding or writing, now’s the time. December 31, 2025 - The Google Nano Banana Pro launch in November 2025 was a next-generation high-performance image generation and manipulation model. It captured the market with its super-detailed visuals, art style control, and speedy inference ...


---
## 6. Build the ReAct Agent

### Step 6.1 — The ReAct Prompt Template

The **prompt template** is the set of instructions we give the LLM that tells it:
- What tools it has available
- What format to follow (Thought / Action / Action Input / Final Answer)
- What the user's question is

The `{placeholders}` in curly braces are filled in automatically by LangChain at runtime:
- `{tools}` — descriptions of all available tools
- `{tool_names}` — just the names of the tools (used to constrain Action choices)
- `{input}` — the user's question
- `{agent_scratchpad}` — the history of Thought/Action/Observation steps so far

In [10]:
REACT_TEMPLATE = """You are a research assistant. Answer the question below using the tools provided.

You have access to the following tools:
{tools}

STRICT RULES — follow these exactly:
1. You MUST call a tool at least once before writing Final Answer.
2. Never answer from memory or training knowledge — always search first.
3. If you are tempted to write Final Answer without searching — STOP and search instead.
4. Use the exact format below, nothing else.

Format:
Question: the input question you must answer
Thought: what do I need to find out?
Action: the tool to use — must be one of [{tool_names}]
Action Input: the exact search query string
Observation: the result returned by the tool
Thought: what did I learn? do I need another search?
Action: (another tool call if needed)
Action Input: (another query if needed)
Observation: (result)
Thought: I now have enough information to answer.
Final Answer: the final answer based on the search results above

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

print("Prompt template ready!")

Prompt template ready!


### Step 6.2 — Build the ReAct Loop

Instead of relying on `AgentExecutor` from LangChain (which has moved across versions), we write the loop ourselves. This is actually **better for learning** because you can see every gear turning.

The loop does exactly what `AgentExecutor` does internally:

```
1. Build the full prompt (question + scratchpad of previous steps)
2. Call the LLM → get raw text output
3. Did the model write "Final Answer"?  →  YES: we're done
4. Parse "Action" and "Action Input" from the output
5. Run the tool with that input → get Observation
6. Append the Observation to the scratchpad and go back to step 1
```

**Key parameters of `run_react_agent`:**
- `question` — the user's question
- `llm` — the LangChain LLM wrapper
- `tools` — list of tool objects the agent can use
- `max_iterations` — safety limit to prevent infinite loops

In [11]:
def run_react_agent(question: str, llm, tools: list, max_iterations: int = 6) -> str:
    """
    A hand-written ReAct loop — works with both HuggingFacePipeline and ChatGroq.
    """
    tool_map   = {t.name: t for t in tools}
    tool_list  = "\n".join([f"- {t.name}: {t.description}" for t in tools])
    tool_names = ", ".join(tool_map.keys())

    scratchpad = ""
    tool_used  = False

    print("=" * 60)
    print(f"Question: {question}")
    print("=" * 60)

    for step in range(1, max_iterations + 1):

        # Step 1: Build the full prompt
        full_prompt = REACT_TEMPLATE.format(
            tools=tool_list,
            tool_names=tool_names,
            input=question,
            agent_scratchpad=scratchpad
        )

        # Step 2: Call the LLM
        # Handle both plain-text LLMs (HuggingFace) and chat LLMs (Groq/OpenAI)
        raw = llm.invoke(full_prompt)
        raw_output = raw.content if hasattr(raw, "content") else str(raw)

        print(f"\n[Step {step}] Model output:\n{raw_output.strip()}")

        # Step 3: Check for Final Answer — only accept it after a tool was used
        if "Final Answer:" in raw_output:
            if not tool_used:
                print("\n  [!] Model tried to answer without searching. Forcing a search...")
                scratchpad += (
                    raw_output.strip()
                    + "\nI need to verify this with a search before answering.\nThought:"
                )
                continue
            answer = raw_output.split("Final Answer:")[-1].strip()
            print("\n" + "=" * 60)
            print(f"FINAL ANSWER: {answer}")
            return answer

        # Step 4: Parse Action and Action Input
        action_match = re.search(r"Action:\s*(.+?)(?:\n|$)", raw_output)
        input_match  = re.search(r"Action Input:\s*(.+?)(?:\n|$)", raw_output)

        if not action_match or not input_match:
            print(f"  [!] Could not parse Action/Action Input at step {step}. Retrying...")
            scratchpad += raw_output + "\n"
            continue

        action_name  = action_match.group(1).strip()
        action_input = input_match.group(1).strip().strip('"\'')

        print(f"\n  > Action      : {action_name}")
        print(f"  > Action Input: {action_input}")

        # Step 5: Run the tool
        if action_name not in tool_map:
            observation = f"Error: unknown tool '{action_name}'. Available: {tool_names}"
        else:
            observation = tool_map[action_name].run(action_input)
            tool_used = True

        print(f"  > Observation : {observation[:300]}...")

        # Step 6: Append to scratchpad and loop
        scratchpad += f"{raw_output.strip()}\nObservation: {observation}\nThought:"

    return "Reached max iterations without a final answer."


print("run_react_agent() ready — works with Gemma (HuggingFace) and Llama (Groq)!")

run_react_agent() ready — works with Gemma (HuggingFace) and Llama (Groq)!


---
## 7. Run the Agent

Now the fun part. We give the agent a question and watch it reason through it step by step.

With `verbose=True`, you will see every `Thought`, `Action`, `Action Input`, and `Observation` printed to the screen. This is what makes agents different from regular LLMs — **the reasoning is visible and auditable**.

### Example 1 — A Current Events Question

This question requires live web search because it's about recent events.

---
## Why Small Models Struggle with ReAct

Gemma 3 4B (4 billion parameters) often fails the ReAct loop for three reasons:

| Problem | What you see |
|---|---|
| **Observation hallucination** | Model writes the `Observation:` line itself from memory instead of calling the tool |
| **Format drift** | As the scratchpad grows, the model loses the Thought/Action structure |
| **Not tool-use trained** | Gemma 4B is a general chat model, not fine-tuned for agent tasks |

### Fix — Switch to Claude 3 Haiku (Anthropic)

**Claude 3 Haiku** is the fastest and most affordable Claude model. It is excellent for agents because:
- Follows the ReAct format reliably — no hallucinated observations
- Very fast responses — ideal for a live demo
- No GPU needed — runs via API, works on any machine

**Steps:**
1. Go to [console.anthropic.com](https://console.anthropic.com) → Sign up → API Keys → Create key
2. Paste your key in the cell below and run it
3. Everything else — tools, prompt, agent loop — stays exactly the same

In [15]:
!pip install -q langchain-anthropic

from langchain_anthropic import ChatAnthropic

# Paste your Anthropic API key here (get it at console.anthropic.com)
ANTHROPIC_API_KEY = ""

llm = ChatAnthropic(
    model="claude-haiku-4-5",
    api_key=ANTHROPIC_API_KEY,
    temperature=0.1,
    stop_sequences=["Observation:"]  # Stop before the model can hallucinate tool results
)

# Quick sanity check
response = llm.invoke("Say hello in one sentence.")
print("Claude says:", response.content)
print("\nLLM switched to Claude 4.5 Haiku — re-run the agent examples below.")

Claude says: Hello! I'm happy to help you with whatever you need.

LLM switched to Claude 4.5 Haiku — re-run the agent examples below.


In [14]:
run_react_agent(
    question="What is the latest version of Python and when was it released?",
    llm=llm,
    tools=[search_tool]
)

Question: What is the latest version of Python and when was it released?

[Step 1] Model output:
Question: What is the latest version of Python and when was it released?

Thought: I need to find out what the current latest version of Python is and when it was released. This is information that changes over time, so I should search for the most current information.

Action: duckduckgo_search

Action Input: latest version of Python release date 2024

  > Action      : duckduckgo_search
  > Action Input: latest version of Python release date 2024
  > Observation : This version currently receives full bug-fix and security updates, while Python 3.13—released in October 2024—will continue to receive bug-fixes until October 2026, and after that will only receive security fixes until its end-of-life in 2029. Python documentation by version. Some previous versions ...

[Step 2] Model output:
Thought: The search results mention Python 3.13 was released in October 2024 and Python 3.14 is listed a

'The latest version of Python is **Python 3.14**, which was released on **October 7, 2025**. According to the search results, Python 3.14 introduced several improvements including clearer syntax error messages and new language features.'

In [16]:
run_react_agent(
    question="latest new on IPL 2026",
    llm=llm,
    tools=[search_tool]
)

Question: latest new on IPL 2026

[Step 1] Model output:
Question: latest news on IPL 2026

Thought: I need to find the most recent news and updates about IPL 2026 (Indian Premier League 2026 season).

Action: duckduckgo_search

Action Input: IPL 2026 latest news

  > Action      : duckduckgo_search
  > Action Input: IPL 2026 latest news
  > Observation : 4 hours ago - The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 ve...

[Step 2] Model output:
Thought: I now have enough information to answer the question about IPL 2026 latest news.

Final Answer: 

Based on the latest news, here are the key updates on IPL 2026:

**Tournament Details:**
- IPL 2026 is the 19th edition of the Indian Premier League, branded as TATA IPL 2026
- The tournament features 10 tea

"Based on the latest news, here are the key updates on IPL 2026:\n\n**Tournament Details:**\n- IPL 2026 is the 19th edition of the Indian Premier League, branded as TATA IPL 2026\n- The tournament features 10 teams competing in 74 matches\n- It is being played from March 28 to May 31 across 13 venues\n\n**Recent Match Updates:**\n- Rajasthan Royals defeated Punjab Kings by 6 wickets in a recent match\n- Donovan Ferreira played a key role with a fiery half-century for Rajasthan Royals\n- This victory ended Punjab Kings' unbeaten run in IPL 2026\n\nThe tournament is currently ongoing with live scores, match updates, and detailed coverage available on the official IPL website and sports platforms."

### Example 2 — A Question That Needs Multiple Searches

This requires the agent to search twice — once for each person — and then combine the information.

In [ ]:
run_react_agent(
    question="Who is the current CEO of OpenAI and when did they take the role?",
    llm=llm,
    tools=[search_tool]
)

### Example 3 — A Factual + Reasoning Question

Here the agent needs to search for data and then do a simple comparison or calculation.

In [ ]:
run_react_agent(
    question="What are the top 3 most popular programming languages in 2025 according to recent surveys?",
    llm=llm,
    tools=[search_tool]
)

---
## 8. Exercises

Now it's your turn! Complete the exercises below.

---

### Exercise 1 — Ask the Agent Something You're Curious About

**Task:** Replace `YOUR QUESTION HERE` with any current-events question you want to know about.

**Examples to try:**
- "What is the current price of Bitcoin?"
- "Who won the most recent cricket match between India and Australia?"
- "What are the new features in the latest iPhone?"

**Observe:** How many search steps does the agent need? Does it get the right answer?

In [ ]:
# Exercise 1: Replace the question below with your own
my_question = "YOUR QUESTION HERE"

# --- Do not change anything below this line ---
run_react_agent(question=my_question, llm=llm, tools=[search_tool])

---
### Exercise 2 — Change the Max Iterations

**Task:** Create a new `AgentExecutor` with `max_iterations=2` and ask it a complex question that requires multiple steps.

**Observe:** What happens when the agent runs out of iterations before it finds the answer? Does it give a partial answer or an error?

**Hint:** Copy the `AgentExecutor(...)` code from above and change one parameter.

In [ ]:
# Exercise 2: Change max_iterations to 2 and observe what happens
run_react_agent(
    question="Compare the population of India and China in 2024 and tell me the difference.",
    llm=llm,
    tools=[search_tool],
    max_iterations=___   # <-- Fill in the value (try 2)
)

---
### Exercise 3 — Add a Second Tool (Custom)

**Task:** Create a custom tool that returns the current date, then add it to the agent's toolbox.

LangChain tools can be created from any Python function using the `@tool` decorator.

**Steps:**
1. Fill in the function body to return today's date as a string
2. Add the new tool to the tools list
3. Create a new `AgentExecutor` with both tools
4. Ask: "What is today's date and who won the FIFA World Cup 2022?"

**Observe:** The agent should use the date tool for the date and the search tool for the World Cup question.

In [ ]:
from datetime import date

# Step 1: Define a tool as a plain Python object with .name, .description, and .run()
class DateTool:
    name = "get_todays_date"
    description = "Returns today's date. Use this when the user asks about the current date."

    def run(self, query: str) -> str:
        # YOUR CODE HERE: return today's date as a string
        return ___

date_tool = DateTool()

# Step 2: Build a list with BOTH tools
tools_v2 = [search_tool, ___]   # <-- add date_tool here

# Step 3: Run the agent with both tools
run_react_agent(
    question="What is today's date and who won the FIFA World Cup 2022?",
    llm=llm,
    tools=tools_v2
)

---
## Summary — What We Built

```
User Question
     |
     v
 AgentExecutor
     |----> ReAct Agent (Gemma 3 4B via HuggingFace)
     |          | Thought: what should I do?
     |          | Action: DuckDuckGoSearchRun
     |          | Action Input: "search query"
     |<---- Observation (search result)
     |----> Agent thinks again...
     |          | Thought: I have enough info.
     |          | Final Answer: ...
     |
     v
 Final Answer to User
```

### Key Concepts Covered
| Concept | What it does |
|---|---|
| **AI Agent** | An LLM that can take actions in a loop |
| **ReAct** | Reasoning + Acting pattern (Thought → Action → Observation) |
| **Tool** | A function the agent can call (e.g. web search) |
| **AgentExecutor** | The loop that runs until a final answer is reached |
| **4-bit Quantization** | Technique to fit large models into limited GPU memory |

### Next Steps to Explore
- Add more tools: Wikipedia, Python REPL, file reader
- Add **memory** so the agent remembers previous conversations
- Try **multi-agent** systems where agents talk to each other
- Explore **LangGraph** for more complex agent workflows